### Modell anzeigen

In [7]:
from core.preprocess import LLM_MODEL, OLLAMA_URL
print("LLM_MODEL =", LLM_MODEL)
print("OLLAMA_URL =", OLLAMA_URL)

LLM_MODEL = gpt-oss:120b-cloud
OLLAMA_URL = http://localhost:11434


### URLs überprüfen, ob sie gültig sind

In [8]:
import requests
import time
import concurrent.futures

BASE_URL = ("https://datenbank2.deutscher-nachhaltigkeitskodex.de/"
            "Profile/MainMenuHandler/2_1?company={company}&year=2024&lang=de&culture=de")

def check_company(session: requests.Session, company_id: int) -> tuple[int, bool]:
    url = BASE_URL.format(company=company_id)
    try:
        resp = session.get(url, timeout=5)
    except requests.RequestException:
        return company_id, False

    if resp.status_code != 200:
        return company_id, False

    if len(resp.text) < 1000:
        return company_id, False

    return company_id, True


def main():
    start_id    = 18780
    end_id      = 100000
    max_workers = 20     # laufen 20 HTTP-Requests gleichzeitig

    t0 = time.perf_counter() # startzeit

    with requests.Session() as session:
        # Optional: Header setzen
        session.headers.update({"User-Agent": "DNK-Scraper/1.0 (kontakt@example.com)"})

        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Wir erzeugen Futures für alle IDs, aber die ThreadPool-Größe begrenzt
            futures = [ executor.submit(check_company, session, cid)
                        for cid in range(start_id, end_id)]

            checked = 0
            for future in concurrent.futures.as_completed(futures):
                company_id, exists = future.result()
                checked += 1

                if exists:
                    print(f"https://datenbank2.deutscher-nachhaltigkeitskodex.de/"
                          f"Profile/MainMenuHandler/2_1?company={company_id}"
                          f"&year=2024&lang=de&culture=de")

            elapsed = time.perf_counter() - t0
            print(f"{checked} URLs überprüft in {elapsed:.3f} s mit {max_workers} Threads")


if __name__ == "__main__":
    main()


https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/2_1?company=18784&year=2024&lang=de&culture=de
https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/2_1?company=18780&year=2024&lang=de&culture=de
81220 URLs überprüft in 1203.604 s mit 20 Threads
